# Deepfake Detection System Inference Notebook

Run this notebook to evaluate the pre-trained model and start the Gradio UI.

In [1]:
!pip install tensorflow opencv-python gradio kagglehub mtcnn retina-face


^C


  Using cached tensorflow-2.21.0-cp311-cp311-win_amd64.whl.metadata (4.5 kB)
  Using cached opencv_python-4.13.0.92-cp37-abi3-win_amd64.whl.metadata (20 kB)
  Using cached kagglehub-1.0.0-py3-none-any.whl.metadata (40 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached flatbuffers-25.12.19-py2.py3-none-any.whl.metadata (1.0 kB)
  Using cached gast-0.7.0-py3-none-any.whl.metadata (1.5 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached termcolor-3.3.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached h5py-3.14.0-cp311-cp311-win_amd64.whl.metadata (2.7 kB)
  Using cached ml_dtypes-0.5.4-cp311-cp311-win_amd64.whl.metadata (9.2 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached url

In [ ]:
import os
import tensorflow as tf
from tensorflow.keras.models import load_model

# Load the model from the backend folder
model_path = r"c:\Users\asus\OneDrive\Desktop\dfds project\backend\deepfake_detector_final.keras"
model = load_model(model_path)
print("Model loaded successfully!")
model.summary()


In [ ]:
import kagglehub
import shutil
import random
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Download dataset
print("Downloading dataset for evaluation...")
path = kagglehub.dataset_download("adham7elmy/deepfake-detection-dataset")
src = os.path.join(path, "dataset")

# 2. Prepare exact evaluation split logic to match 96% accuracy
real_images = os.listdir(os.path.join(src, "real"))
fake_images = os.listdir(os.path.join(src, "fake"))

min_count = min(len(real_images), len(fake_images))

# Use fixed seed to recreate the same split
random.seed(42)
real_sample = random.sample(real_images, min_count)
fake_sample = random.sample(fake_images, min_count)

split_dir = "split_data"
os.makedirs(os.path.join(split_dir, "test", "real"), exist_ok=True)
os.makedirs(os.path.join(split_dir, "test", "fake"), exist_ok=True)

# 85% was used for train/val, 15% for test
test_start = int(0.85 * min_count)

for img in real_sample[test_start:]:
    shutil.copy(os.path.join(src, "real", img), os.path.join(split_dir, "test", "real", img))

for img in fake_sample[test_start:]:
    shutil.copy(os.path.join(src, "fake", img), os.path.join(split_dir, "test", "fake", img))

print("Test dataset prepared.")

# 3. Evaluate
test_gen = ImageDataGenerator(rescale=1./255)
test_data = test_gen.flow_from_directory(
    os.path.join(split_dir, "test"),
    target_size=(224,224),
    batch_size=16,
    class_mode='binary'
)

loss, acc, auc = model.evaluate(test_data)
print(f"\nFINAL TEST ACCURACY: {acc * 100:.2f}%")
print(f"FINAL TEST AUC: {auc:.4f}")


In [ ]:
import gradio as gr
import numpy as np
import cv2

def predict_image(image):
    image = np.array(image)
    # Handle RGBA images
    if image.shape[-1] == 4:
        image = image[:, :, :3]
    
    image = cv2.resize(image, (224, 224))
    image = image / 255.0
    image = np.expand_dims(image, axis=0)
    
    prediction = model.predict(image, verbose=0)[0][0]
    
    if prediction >= 0.5:
        return f"REAL (Confidence: {prediction:.4f})"
    else:
        return f"FAKE (Confidence: {1 - prediction:.4f})"

gr.close_all()
interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil", label="Upload Image"),
    outputs=gr.Textbox(label="Prediction"),
    title="Deepfake Detection System",
    description="Upload an image to check whether it is REAL or FAKE."
)

print("\nLaunching Gradio Interface...")
interface.launch(share=False)
